# RizeHire: Enhanced BERT + ANN Model V2 for Resume Classification
**Sandeep Kumar (2022UG1113) & Nikhil Sonkar (2022UG1111)**  
**IIIT Ranchi | Major Project | 2026**

### Key Improvements over V1 (57% → Target 85%+):
1. **Cross-Encoder similarity scores** as primary features (captures cross-attention)
2. **Longer text input** (2048 chars instead of 512)
3. **Transcript column** used for additional signal
4. **Rich feature engineering** — cosine similarity, element-wise interactions, text statistics
5. **Class-weighted loss** to handle imbalanced data
6. **Residual ANN** with skip connections, GELU activation, and better regularization
7. **Lower learning rate** with cosine annealing scheduler

**Pipeline:** Raw Text → BERT Cross-Encoder Scores + SBERT Embeddings + Text Features → Feature Fusion (1163-dim) → Residual ANN → Classification

---

## Step 1: Setup & Install Dependencies

In [ ]:
!pip install -q sentence-transformers torch scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    try:
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU Memory: {gpu_mem:.1f} GB')
    except Exception:
        print('GPU detected successfully')

## Step 2: Load Dataset

In [ ]:
# Upload your dataset.csv
from google.colab import files
import pandas as pd

print("Upload your dataset.csv file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"\nLoaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Dataset columns: ID, Name, Role, Transcript, Resume, decision, Reason_for_decision, Job_Description
resume_col = 'Resume'
job_col = 'Job_Description'
transcript_col = 'Transcript'
label_col = 'decision'

# Drop rows with missing critical fields
df = df.dropna(subset=[resume_col, job_col, label_col])
print(f"After dropping NaN: {len(df)} rows")

# Analyze class distribution
print(f"\nLabel distribution:")
print(df[label_col].value_counts())
print(f"\nLabel proportions:")
print(df[label_col].value_counts(normalize=True).round(3))

# Encode labels
le = LabelEncoder()
labels = le.fit_transform(df[label_col])
num_classes = len(le.classes_)
print(f"\nEncoded classes: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Number of classes: {num_classes}")

## Step 3: Text Preprocessing & Feature Engineering

In [ ]:
def clean_text(text):
    """Clean and normalize text"""
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_text_features(resume, job_desc, transcript=''):
    """Extract handcrafted text features for better signal"""
    resume = str(resume).lower()
    job_desc = str(job_desc).lower()
    transcript = str(transcript).lower() if transcript else ''
    
    resume_words = set(resume.split())
    job_words = set(job_desc.split())
    
    # Common skill keywords
    tech_keywords = {'python', 'java', 'javascript', 'react', 'node', 'sql', 'aws', 
                     'docker', 'kubernetes', 'machine', 'learning', 'deep', 'neural',
                     'tensorflow', 'pytorch', 'api', 'rest', 'git', 'agile', 'scrum',
                     'html', 'css', 'angular', 'vue', 'mongodb', 'postgresql', 'redis',
                     'linux', 'cloud', 'azure', 'gcp', 'ci', 'cd', 'devops', 'data',
                     'analytics', 'tableau', 'power', 'bi', 'excel', 'spark', 'hadoop',
                     'nlp', 'cv', 'bert', 'transformer', 'microservices', 'kafka'}
    
    # Soft skill keywords  
    soft_keywords = {'leadership', 'communication', 'teamwork', 'problem', 'solving',
                     'analytical', 'creative', 'management', 'collaboration', 'initiative',
                     'adaptable', 'detail', 'organized', 'motivated', 'proactive'}
    
    features = [
        # Length features
        len(resume) / 10000,  # normalized resume length
        len(job_desc) / 5000,  # normalized job desc length
        len(transcript) / 10000,  # normalized transcript length
        len(resume.split()),  # word count resume
        len(job_desc.split()),  # word count job
        
        # Overlap features
        len(resume_words & job_words) / max(len(job_words), 1),  # keyword overlap ratio
        len(resume_words & job_words) / max(len(resume_words), 1),  # reverse overlap
        
        # Technical skill matches
        len(resume_words & tech_keywords),  # tech skills in resume
        len(job_words & tech_keywords),  # tech skills in job
        len(resume_words & job_words & tech_keywords) / max(len(job_words & tech_keywords), 1),  # tech match rate
        
        # Soft skill matches
        len(resume_words & soft_keywords),
        len(resume_words & job_words & soft_keywords) / max(len(job_words & soft_keywords), 1),
        
        # Transcript features (interview signal)
        1.0 if len(transcript) > 100 else 0.0,  # has substantial transcript
        len(set(transcript.split()) & tech_keywords) if transcript else 0,  # tech mentions in interview
    ]
    return features

# Use LONGER text (2048 chars instead of 512)
MAX_TEXT_LEN = 2048

resume_texts = df[resume_col].apply(clean_text).apply(lambda x: x[:MAX_TEXT_LEN]).tolist()
job_texts = df[job_col].apply(clean_text).apply(lambda x: x[:MAX_TEXT_LEN]).tolist()
transcript_texts = df[transcript_col].apply(clean_text).apply(lambda x: x[:MAX_TEXT_LEN]).tolist() if transcript_col in df.columns else [''] * len(df)

# Extract handcrafted features
print("Extracting text features...")
text_features = []
for i in range(len(df)):
    trans = transcript_texts[i] if i < len(transcript_texts) else ''
    text_features.append(extract_text_features(resume_texts[i], job_texts[i], trans))

text_features = np.array(text_features, dtype=np.float32)
print(f"Text features shape: {text_features.shape}")
print(f"Resume text max length used: {MAX_TEXT_LEN} chars")
print(f"Sample feature vector: {text_features[0]}")

## Step 4: Generate BERT Embeddings (SBERT + Cross-Encoder)

**Key improvement:** We use BOTH:
- **SBERT (bi-encoder)** — individual semantic embeddings (384-dim each)
- **Cross-Encoder** — direct resume↔job interaction score (captures cross-attention)
- **Interaction features** — cosine similarity, element-wise product/difference

In [ ]:
# Load SBERT model for embeddings
print("Loading Sentence-BERT model (all-MiniLM-L6-v2)...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
embed_dim = sbert_model.get_sentence_embedding_dimension()
print(f"SBERT loaded! Embedding dim: {embed_dim}")

# Load Cross-Encoder for similarity scoring
print("\nLoading Cross-Encoder (ms-marco-MiniLM-L-6-v2)...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
print("Cross-Encoder loaded!")

In [ ]:
# Generate SBERT embeddings for resume texts
print("Generating resume embeddings...")
resume_embeddings = sbert_model.encode(resume_texts, batch_size=64, 
                                       show_progress_bar=True, convert_to_numpy=True)
print(f"Resume embeddings: {resume_embeddings.shape}")

# Generate SBERT embeddings for job description texts
print("\nGenerating job description embeddings...")
job_embeddings = sbert_model.encode(job_texts, batch_size=64,
                                    show_progress_bar=True, convert_to_numpy=True)
print(f"Job embeddings: {job_embeddings.shape}")

# Generate SBERT embeddings for transcript texts (additional signal)
print("\nGenerating transcript embeddings...")
transcript_embeddings = sbert_model.encode(transcript_texts, batch_size=64,
                                           show_progress_bar=True, convert_to_numpy=True)
print(f"Transcript embeddings: {transcript_embeddings.shape}")

In [ ]:
# Generate Cross-Encoder scores (resume <-> job_description)
# This is the KEY improvement — cross-encoder sees both texts together
print("Computing Cross-Encoder similarity scores...")
print("(This captures cross-attention between resume and job description)")

# Prepare pairs for cross-encoder
pairs_resume_job = [[resume_texts[i], job_texts[i]] for i in range(len(resume_texts))]

# Score in batches
cross_scores_rj = cross_encoder.predict(pairs_resume_job, batch_size=64, show_progress_bar=True)
cross_scores_rj = np.array(cross_scores_rj, dtype=np.float32).reshape(-1, 1)
print(f"Cross-encoder scores (resume↔job): {cross_scores_rj.shape}")
print(f"Score range: [{cross_scores_rj.min():.3f}, {cross_scores_rj.max():.3f}]")
print(f"Mean score: {cross_scores_rj.mean():.3f}")

# Also compute transcript <-> job similarity (interview relevance)
print("\nComputing Cross-Encoder scores (transcript↔job)...")
pairs_trans_job = [[transcript_texts[i], job_texts[i]] for i in range(len(transcript_texts))]
cross_scores_tj = cross_encoder.predict(pairs_trans_job, batch_size=64, show_progress_bar=True)
cross_scores_tj = np.array(cross_scores_tj, dtype=np.float32).reshape(-1, 1)
print(f"Cross-encoder scores (transcript↔job): {cross_scores_tj.shape}")

In [ ]:
# Compute interaction features between embeddings
print("Computing embedding interaction features...")

# Cosine similarity between resume and job embeddings
from numpy.linalg import norm
cosine_sim_rj = np.sum(resume_embeddings * job_embeddings, axis=1) / (
    norm(resume_embeddings, axis=1) * norm(job_embeddings, axis=1) + 1e-8)
cosine_sim_rj = cosine_sim_rj.reshape(-1, 1)

# Cosine similarity between resume and transcript
cosine_sim_rt = np.sum(resume_embeddings * transcript_embeddings, axis=1) / (
    norm(resume_embeddings, axis=1) * norm(transcript_embeddings, axis=1) + 1e-8)
cosine_sim_rt = cosine_sim_rt.reshape(-1, 1)

# Element-wise product (captures matching dimensions)
elem_product_rj = resume_embeddings * job_embeddings  # 384-dim

# Element-wise absolute difference (captures mismatches)
elem_diff_rj = np.abs(resume_embeddings - job_embeddings)  # 384-dim

print(f"Cosine sim (resume↔job): {cosine_sim_rj.shape}")
print(f"Cosine sim (resume↔transcript): {cosine_sim_rt.shape}")
print(f"Element-wise product: {elem_product_rj.shape}")
print(f"Element-wise diff: {elem_diff_rj.shape}")

In [ ]:
# COMBINE ALL FEATURES
# Strategy: Use interaction features (more discriminative) + cross-encoder scores + text features
# Instead of raw 384+384 concatenation which loses cross-interaction

combined_features = np.concatenate([
    elem_product_rj,        # 384-dim: element-wise resume*job
    elem_diff_rj,           # 384-dim: |resume - job|
    cross_scores_rj,        # 1-dim: cross-encoder resume↔job score
    cross_scores_tj,        # 1-dim: cross-encoder transcript↔job score
    cosine_sim_rj,          # 1-dim: cosine similarity resume↔job
    cosine_sim_rt,          # 1-dim: cosine similarity resume↔transcript
    text_features,          # 14-dim: handcrafted text features
], axis=1)

print(f"\n{'='*60}")
print(f"FINAL COMBINED FEATURE VECTOR: {combined_features.shape}")
print(f"{'='*60}")
print(f"  Element-wise product (resume*job):   384 dims")
print(f"  Element-wise |resume - job|:         384 dims")
print(f"  Cross-encoder score (resume↔job):      1 dim")
print(f"  Cross-encoder score (transcript↔job):  1 dim")
print(f"  Cosine similarity (resume↔job):        1 dim")
print(f"  Cosine similarity (resume↔transcript): 1 dim")
print(f"  Handcrafted text features:            14 dims")
print(f"  ─────────────────────────────────────────")
print(f"  TOTAL:                               {combined_features.shape[1]} dims")

## Step 5: Normalize Features & Split Data

In [ ]:
# Normalize features (important for ANN performance)
scaler = StandardScaler()
combined_features_scaled = scaler.fit_transform(combined_features)
print(f"Features normalized with StandardScaler")
print(f"Mean after scaling: {combined_features_scaled.mean():.6f}")
print(f"Std after scaling: {combined_features_scaled.std():.6f}")

# Split: 80% train, 10% val, 10% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    combined_features_scaled, labels, test_size=0.1, random_state=42, stratify=labels)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.111, random_state=42, stratify=y_train_val)

print(f"\nTraining set:   {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")

# Compute class weights for imbalanced data
class_counts = Counter(y_train)
total = len(y_train)
class_weights = {cls: total / (num_classes * count) for cls, count in class_counts.items()}
print(f"\nClass distribution in training set: {dict(class_counts)}")
print(f"Class weights: {class_weights}")

# Create weight tensor for loss function
weight_tensor = torch.FloatTensor([class_weights[i] for i in range(num_classes)]).to(device)
print(f"Loss weights: {weight_tensor}")

In [ ]:
# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.LongTensor(y_val).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

# Weighted random sampler for training (handles class imbalance)
sample_weights = [class_weights[y] for y in y_train]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler)

val_dataset = TensorDataset(X_val_t, y_val_t)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Batch size: 64")
print(f"Training batches per epoch: {len(train_loader)}")
print(f"Using WeightedRandomSampler for class balancing")

## Step 6: Define Enhanced Residual ANN Architecture

**Architecture:** Input → [FC+BN+GELU+Dropout] with Residual Skip Connections → Output  
Key improvements: GELU activation, skip connections, label smoothing

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with skip connection"""
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.dropout(self.activation(x + self.block(x)))


class EnhancedResumeClassifierANN(nn.Module):
    """
    Enhanced ANN with Residual Connections for Resume Classification
    
    Architecture:
    Input (786-dim) → Projection (512) → ResBlock → ResBlock → FC(256) → FC(128) → Output
    
    Improvements:
    - GELU activation (smoother gradients than ReLU)
    - Residual/skip connections (better gradient flow)
    - Layer normalization + BatchNorm
    - Graduated dropout (higher in early layers)
    """
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
        )
        
        # Residual blocks (skip connections help with gradient flow)
        self.res_block1 = ResidualBlock(512, dropout=0.3)
        self.res_block2 = ResidualBlock(512, dropout=0.25)
        
        # Reduction layers
        self.reduce = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.15),
        )
        
        # Output head
        self.classifier = nn.Linear(128, num_classes)
    
    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_block1(x)
        x = self.res_block2(x)
        x = self.reduce(x)
        return self.classifier(x)


input_dim = combined_features_scaled.shape[1]
model = EnhancedResumeClassifierANN(input_dim=input_dim, num_classes=num_classes).to(device)

print(f"Enhanced Residual ANN Architecture:")
print(f"Input dimension: {input_dim}")
print(f"Output classes: {num_classes}")
print(f"\n{model}")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Step 7: Train the Enhanced Model

In [ ]:
# Loss with class weights + label smoothing
criterion = nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=0.1)

# AdamW optimizer (better weight decay handling)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-3)

# Cosine annealing scheduler (smoother LR decay)
EPOCHS = 80
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

best_val_acc = 0
best_val_f1 = 0
patience = 20
patience_counter = 0

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

print(f"Training Enhanced BERT+ANN V2 for {EPOCHS} epochs...")
print(f"Optimizer: AdamW (lr=0.0005, weight_decay=1e-3)")
print(f"Scheduler: CosineAnnealingWarmRestarts (T_0=20)")
print(f"Loss: CrossEntropy with class weights + label smoothing (0.1)")
print(f"Early stopping patience: {patience}")
print("=" * 80)

for epoch in range(EPOCHS):
    # ── Training ──
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += batch_y.size(0)
        train_correct += (predicted == batch_y).sum().item()

    scheduler.step(epoch)  # pass epoch number for CosineAnnealingWarmRestarts
    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    # ── Validation ──
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    all_val_preds = []
    all_val_labels = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_labels.extend(batch_y.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    val_f1 = f1_score(all_val_labels, all_val_preds, average='weighted')

    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # Save best model (by F1 score, more robust than accuracy)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_bert_ann_v2.pth')
        patience_counter = 0
        marker = ' *** BEST (F1: {:.2f}%)'.format(val_f1 * 100)
    else:
        patience_counter += 1
        marker = ''

    if (epoch + 1) % 5 == 0 or epoch == 0 or marker:
        print("Epoch [{:3d}/{}] | Train: {:.2f}% (loss:{:.4f}) | Val: {:.2f}% (loss:{:.4f}) | LR: {:.6f}{}".format(
            epoch + 1, EPOCHS, train_acc * 100, train_loss, val_acc * 100, val_loss, current_lr, marker))

    if patience_counter >= patience:
        print("\nEarly stopping at epoch {}!".format(epoch + 1))
        break

print("\n" + "=" * 80)
print("Training complete! Best val accuracy: {:.2f}% | Best val F1: {:.2f}%".format(
    best_val_acc * 100, best_val_f1 * 100))

## Step 8: Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_acc']) + 1)

# Accuracy
axes[0].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-', linewidth=2, label='Training', alpha=0.8)
axes[0].plot(epochs_range, [a*100 for a in history['val_acc']], 'g-', linewidth=2, label='Validation', alpha=0.8)
axes[0].axhline(y=best_val_acc*100, color='r', linestyle='--', alpha=0.7, label=f'Best: {best_val_acc*100:.2f}%')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('BERT + Residual ANN — Accuracy', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(epochs_range, history['train_loss'], 'b-', linewidth=2, label='Training', alpha=0.8)
axes[1].plot(epochs_range, history['val_loss'], 'r-', linewidth=2, label='Validation', alpha=0.8)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('BERT + Residual ANN — Loss', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Learning Rate
axes[2].plot(epochs_range, history['lr'], 'purple', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('Cosine Annealing LR Schedule', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bert_ann_training_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_training_curves.png")

## Step 9: Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_bert_ann_v2.pth', weights_only=True))
model.eval()

with torch.no_grad():
    test_outputs = model(X_test_t)
    _, y_pred = torch.max(test_outputs, 1)
    probs = torch.softmax(test_outputs, dim=1).cpu().numpy()

y_pred_np = y_pred.cpu().numpy()
y_test_np = y_test_t.cpu().numpy()

test_acc = accuracy_score(y_test_np, y_pred_np)
test_prec = precision_score(y_test_np, y_pred_np, average='weighted')
test_rec = recall_score(y_test_np, y_pred_np, average='weighted')
test_f1 = f1_score(y_test_np, y_pred_np, average='weighted')

print("=" * 60)
print("   BERT + RESIDUAL ANN (V2) — TEST SET RESULTS")
print("=" * 60)
print("   Accuracy:  {:.2f}%".format(test_acc * 100))
print("   Precision: {:.2f}%".format(test_prec * 100))
print("   Recall:    {:.2f}%".format(test_rec * 100))
print("   F1-Score:  {:.2f}%".format(test_f1 * 100))
print("=" * 60)

# Improvement over V1
v1_acc = 56.58
improvement = test_acc * 100 - v1_acc
print("\nImprovement over V1: {:.2f}% -> {:.2f}% (+{:.2f}%)".format(v1_acc, test_acc * 100, improvement))

print("\nDetailed Classification Report:")
print(classification_report(y_test_np, y_pred_np))

## Step 10: Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test_np, y_pred_np)
if num_classes == 2:
    tick_labels = list(le.classes_) if hasattr(le, 'classes_') else ['Rejected', 'Accepted']
else:
    tick_labels = list(le.classes_) if hasattr(le, 'classes_') else list(range(num_classes))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=tick_labels, yticklabels=tick_labels)
ax1.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax1.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax1.set_title(f'Confusion Matrix (Acc: {test_acc*100:.2f}%)', fontsize=13, fontweight='bold')

# ROC Curve
if num_classes == 2:
    fpr, tpr, _ = roc_curve(y_test_np, probs[:, 1])
    roc_auc = auc(fpr, tpr)
    ax2.plot(fpr, tpr, 'b-', linewidth=2, label=f'BERT+ResANN V2 (AUC = {roc_auc:.3f})')
    ax2.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Baseline')
    ax2.set_xlabel('False Positive Rate', fontsize=12)
    ax2.set_ylabel('True Positive Rate', fontsize=12)
    ax2.set_title('ROC Curve', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.fill_between(fpr, tpr, alpha=0.1, color='blue')
else:
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    colors = plt.cm.Blues(np.linspace(0.4, 0.8, num_classes))
    bars = ax2.bar(range(num_classes), per_class_acc * 100, color=colors)
    ax2.set_xlabel('Class', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Per-Class Accuracy', fontsize=13, fontweight='bold')
    ax2.set_xticks(range(num_classes))
    ax2.set_xticklabels(tick_labels, rotation=45, ha='right')
    for bar, acc in zip(bars, per_class_acc):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc*100:.1f}%', ha='center', fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('bert_ann_evaluation.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_evaluation.png")

## Step 11: Pipeline Flow Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')

# Colors
c_input = '#3498DB'    # blue
c_bert = '#E74C3C'     # red
c_feature = '#2ECC71'  # green
c_ann = '#9B59B6'      # purple
c_output = '#F39C12'   # orange

def draw_box(ax, x, y, w, h, text, color, fontsize=9):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='white',
                         alpha=0.85, linewidth=2, zorder=2, joinstyle='round')
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', zorder=3,
            multialignment='center')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

# Title
ax.text(8, 9.5, 'RizeHire: BERT + Residual ANN Pipeline (V2)', ha='center',
        fontsize=16, fontweight='bold', color='#2C3E50')

# Row 1: Input Data
draw_box(ax, 0.5, 7.8, 2.5, 0.8, 'Resume\n(2048 chars)', c_input)
draw_box(ax, 3.5, 7.8, 2.5, 0.8, 'Job Description\n(2048 chars)', c_input)
draw_box(ax, 6.5, 7.8, 2.5, 0.8, 'Interview\nTranscript', c_input)
draw_box(ax, 10, 7.8, 2.5, 0.8, 'Text Statistics\n(14 features)', c_input)

# Arrows down
draw_arrow(ax, 1.75, 7.8, 1.75, 7.0)
draw_arrow(ax, 4.75, 7.8, 4.75, 7.0)
draw_arrow(ax, 7.75, 7.8, 7.75, 7.0)
draw_arrow(ax, 11.25, 7.8, 11.25, 7.0)

# Row 2: BERT Processing
draw_box(ax, 0.5, 6.2, 2.5, 0.8, 'SBERT Encoder\n(384-dim)', c_bert)
draw_box(ax, 3.5, 6.2, 2.5, 0.8, 'SBERT Encoder\n(384-dim)', c_bert)
draw_box(ax, 6.5, 6.2, 2.5, 0.8, 'SBERT Encoder\n(384-dim)', c_bert)
draw_box(ax, 10, 6.2, 2.5, 0.8, 'Cross-Encoder\n(Similarity Score)', c_bert)

# Arrows down
draw_arrow(ax, 1.75, 6.2, 3.0, 5.5)
draw_arrow(ax, 4.75, 6.2, 4.75, 5.5)
draw_arrow(ax, 7.75, 6.2, 6.5, 5.5)
draw_arrow(ax, 11.25, 6.2, 10.0, 5.5)

# Row 3: Feature Engineering
draw_box(ax, 1.5, 4.7, 3.0, 0.8, 'Element-wise Product\nResume * Job (384-dim)', c_feature, fontsize=8)
draw_box(ax, 5.0, 4.7, 3.0, 0.8, '|Resume - Job|\nAbsolute Diff (384-dim)', c_feature, fontsize=8)
draw_box(ax, 8.5, 4.7, 2.5, 0.8, 'Cosine Similarity\n+ Cross-Encoder', c_feature, fontsize=8)
draw_box(ax, 11.5, 4.7, 2.5, 0.8, 'Text Features\n(14-dim)', c_feature, fontsize=8)

# Arrows to concatenation
draw_arrow(ax, 3.0, 4.7, 6.5, 4.0)
draw_arrow(ax, 6.5, 4.7, 7.0, 4.0)
draw_arrow(ax, 9.75, 4.7, 8.5, 4.0)
draw_arrow(ax, 12.75, 4.7, 9.0, 4.0)

# Row 4: Concatenation
concat_text = 'Feature Fusion (Concatenation)\n{}-dim vector -> StandardScaler'.format(input_dim)
draw_box(ax, 5.0, 3.2, 5.5, 0.8, concat_text, c_feature)

# Arrow down
draw_arrow(ax, 7.75, 3.2, 7.75, 2.6)

# Row 5: ANN
draw_box(ax, 3.5, 1.8, 8.5, 0.8,
         'Residual ANN: 512 -> ResBlock x2 -> 256 -> 128 -> Softmax\n(GELU, BatchNorm, Dropout, Skip Connections, Gradient Clipping)',
         c_ann, fontsize=9)

# Arrow down
draw_arrow(ax, 7.75, 1.8, 7.75, 1.2)

# Row 6: Output
output_text = 'Classification Output\nAccuracy: {:.2f}% | F1: {:.2f}%'.format(test_acc * 100, test_f1 * 100)
draw_box(ax, 5.5, 0.4, 4.5, 0.8, output_text, c_output)

plt.tight_layout()
plt.savefig('pipeline_flow_v2.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: pipeline_flow_v2.png")

## Step 12: Research Paper Comparison

In [ ]:
# Comparison with latest research papers from our literature survey
comparison_data = {
    'Study': [
        'Sabarirajan et al. (2025)',
        'Deshmukh & Raut (2024)',
        'Wankhede et al. (2024)',
        'More & Kene (2024)',
        'Jyothsna et al. (2024)',
        'Bevara et al. (2025)',
        'Sriranjani & Kalaiarasi (2025)',
        'Wilson & Caliskan (2025)',
        'Lo (2025)',
        'Tejaswini et al. (2022)',
        'RizeHire V1 (Ours)',
        'RizeHire V2 (Ours)'
    ],
    'Technique': [
        'BERT+RoBERTa Ensemble',
        'NLP+Hybrid ML',
        'NLP+Decision Tree',
        'TF-IDF+SVM/RF',
        'Cosine+AdaBoost',
        'SBERT Embeddings',
        'BERT+Random Forest',
        'GPT-4 Zero-Shot',
        'BERT Fine-Tuning',
        'TF-IDF+ML Pipeline',
        'SBERT+ANN (concat)',
        'CrossEncoder+SBERT+ResANN'
    ],
    'Accuracy': [
        87, 88, 96, 90, 87, 78, 87, 82, 85, 85,
        56.58,
        round(test_acc * 100, 2)
    ],
    'Has_XAI': [
        True, False, False, False, False, False, False, False, False, False,
        True, True
    ],
    'Real_Time': [
        False, False, False, False, False, False, False, False, False, False,
        True, True
    ]
}

comp_df = pd.DataFrame(comparison_data)

fig, ax = plt.subplots(figsize=(14, 7))

colors = []
for i, row in comp_df.iterrows():
    if 'V2' in row['Study']:
        colors.append('#27AE60')  # green for V2
    elif 'V1' in row['Study']:
        colors.append('#E74C3C')  # red for V1
    else:
        colors.append('#95A5A6')  # gray for others

bars = ax.barh(comp_df['Study'], comp_df['Accuracy'], color=colors, 
               edgecolor='white', height=0.6)

ax.set_xlabel('Accuracy / F1-Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Performance Comparison with Latest Research Papers (2022-2025)', 
             fontsize=14, fontweight='bold')
ax.set_xlim(0, 110)
ax.grid(True, alpha=0.3, axis='x')

for bar, acc, xai, rt in zip(bars, comp_df['Accuracy'], comp_df['Has_XAI'], comp_df['Real_Time']):
    label = f'{acc}%'
    extras = []
    if xai: extras.append('XAI')
    if rt: extras.append('Real-Time')
    if extras: label += f' + {" + ".join(extras)}'
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, label,
            va='center', fontweight='bold', fontsize=9)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#27AE60', label='RizeHire V2 (Current)'),
    Patch(facecolor='#E74C3C', label='RizeHire V1 (Previous)'),
    Patch(facecolor='#95A5A6', label='Other Studies'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('bert_ann_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_comparison.png")
print("\nComparison Table:")
print(comp_df.to_string(index=False))

## Step 13: SHAP Explainability

In [ ]:
!pip install -q shap

In [ ]:
import shap

def model_predict(X):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X).to(device)
        outputs = torch.softmax(model(X_tensor), dim=1)
        return outputs.cpu().numpy()

# Feature names for interpretability
feature_names = (
    [f'Resume×Job_{i}' for i in range(384)] +
    [f'|Res-Job|_{i}' for i in range(384)] +
    ['CrossEnc_Resume↔Job', 'CrossEnc_Trans↔Job',
     'CosSim_Resume↔Job', 'CosSim_Resume↔Trans'] +
    ['ResumeLen', 'JobLen', 'TranscriptLen', 'ResumeWords', 'JobWords',
     'KeywordOverlap', 'ReverseOverlap', 'TechSkillsResume', 'TechSkillsJob',
     'TechMatchRate', 'SoftSkillsResume', 'SoftSkillMatch',
     'HasTranscript', 'TechInInterview']
)

# SHAP analysis
background = X_train[:50]  # small background for speed
explainer = shap.KernelExplainer(model_predict, background)

test_samples = X_test[:15]
print("Computing SHAP values (this may take a few minutes)...")
shap_values = explainer.shap_values(test_samples)
print("SHAP values computed!")

# Focus on the most interpretable features (last 18: cross-encoder + text features)
important_idx = list(range(768, input_dim))  # cross-encoder scores + text features
important_names = [feature_names[i] for i in important_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Full SHAP summary (top 20 features)
plt.sca(ax1)
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], test_samples, feature_names=feature_names,
                      max_display=20, show=False)
else:
    shap.summary_plot(shap_values, test_samples, feature_names=feature_names,
                      max_display=20, show=False)
ax1.set_title('SHAP — Top 20 Features', fontsize=12, fontweight='bold')

# Interpretable features only
plt.sca(ax2)
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1][:, important_idx], test_samples[:, important_idx],
                      feature_names=important_names, max_display=18, show=False)
else:
    shap.summary_plot(shap_values[:, important_idx], test_samples[:, important_idx],
                      feature_names=important_names, max_display=18, show=False)
ax2.set_title('SHAP — Interpretable Features', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('bert_ann_shap.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_shap.png")

## Step 14: Feature Importance Analysis

In [ ]:
# Analyze which feature GROUPS matter most
if isinstance(shap_values, list):
    sv = np.abs(shap_values[1])
else:
    sv = np.abs(shap_values)

# Group importance
groups = {
    'Resume×Job (element-wise)': sv[:, :384].mean(),
    '|Resume-Job| (difference)': sv[:, 384:768].mean(),
    'Cross-Encoder Scores': sv[:, 768:770].mean(),
    'Cosine Similarities': sv[:, 770:772].mean(),
    'Text Statistics': sv[:, 772:].mean(),
}

fig, ax = plt.subplots(figsize=(10, 5))
group_names = list(groups.keys())
group_values = list(groups.values())
colors = ['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6', '#F39C12']

bars = ax.barh(group_names, group_values, color=colors, edgecolor='white', height=0.5)
ax.set_xlabel('Mean |SHAP Value|', fontsize=12, fontweight='bold')
ax.set_title('Feature Group Importance in BERT+ANN V2', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, group_values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('feature_group_importance.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: feature_group_importance.png")

## Step 15: Save Model & Download All Results

In [ ]:
# Save final model with all metadata
save_dict = {
    'model_state_dict': model.state_dict(),
    'input_dim': input_dim,
    'num_classes': num_classes,
    'architecture': '{}->512->ResBlockx2->256->128->{}'.format(input_dim, num_classes),
    'test_accuracy': test_acc,
    'test_precision': test_prec,
    'test_recall': test_rec,
    'test_f1': test_f1,
    'best_val_accuracy': best_val_acc,
    'best_val_f1': best_val_f1,
    'training_history': history,
    'label_encoder_classes': list(le.classes_),
    'feature_description': {
        'elem_product': '384-dim (resume * job embedding)',
        'elem_diff': '384-dim (|resume - job| embedding)',
        'cross_encoder_rj': '1-dim (cross-encoder resume<->job)',
        'cross_encoder_tj': '1-dim (cross-encoder transcript<->job)',
        'cosine_sim_rj': '1-dim (cosine resume<->job)',
        'cosine_sim_rt': '1-dim (cosine resume<->transcript)',
        'text_features': '14-dim (handcrafted)',
    },
    'v1_accuracy': 56.58,
    'improvement': test_acc * 100 - 56.58,
}
torch.save(save_dict, 'rizehire_bert_ann_v2_final.pth')

print("Model saved: rizehire_bert_ann_v2_final.pth")

# Final summary
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print("\n" + "=" * 70)
print("       FINAL RESULTS SUMMARY — RizeHire BERT+ANN V2")
print("=" * 70)
print("  BERT Models:      SBERT (all-MiniLM-L6-v2) + Cross-Encoder (ms-marco)")
print("  ANN Architecture: {}->512->ResBlockx2->256->128->{}".format(input_dim, num_classes))
print("  Features:         Interaction features + Cross-encoder + Text stats")
print("  Training Samples: {}".format(len(X_train)))
print("  Test Samples:     {}".format(len(X_test)))
print("  GPU:              {}".format(gpu_name))
print("  " + "-" * 45)
print("  Test Accuracy:    {:.2f}%".format(test_acc * 100))
print("  Test Precision:   {:.2f}%".format(test_prec * 100))
print("  Test Recall:      {:.2f}%".format(test_rec * 100))
print("  Test F1-Score:    {:.2f}%".format(test_f1 * 100))
print("  Best Val Acc:     {:.2f}%".format(best_val_acc * 100))
print("  Best Val F1:      {:.2f}%".format(best_val_f1 * 100))
print("  " + "-" * 45)
print("  V1 Accuracy:      56.58%")
print("  V2 Accuracy:      {:.2f}%".format(test_acc * 100))
print("  Improvement:      +{:.2f}%".format(test_acc * 100 - 56.58))
print("=" * 70)

In [ ]:
# Download all results
from google.colab import files

download_files = [
    'rizehire_bert_ann_v2_final.pth',
    'bert_ann_training_curves.png',
    'bert_ann_evaluation.png',
    'bert_ann_comparison.png',
    'bert_ann_shap.png',
    'pipeline_flow_v2.png',
    'feature_group_importance.png',
]

print("Downloading all result files...")
for f in download_files:
    try:
        files.download(f)
        print(f"  ✓ {f}")
    except:
        print(f"  ✗ {f} — download manually from file browser")

print("\n✅ Done! Share the PNG files with your professor.")
print("The .pth file contains your trained model weights.")